# Deep learning on images

## Load modules from repo

In [1]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [2]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [3]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features, load_preprocessors, save_preprocessors
from src.models.on_images.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

2025-10-04 15:16:04.127170: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-04 15:16:04.160319: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-04 15:16:05.390728: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1759583766.341872   52759 gpu_device.cc:2020] Created device /job:localhost/rep

In [4]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)
importlib.reload(src.models.on_images.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

<module 'src.models.on_text_and_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_text_and_images/deep_learning.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')
full_y_train=y_train

In [9]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [ ]:
version=1
artifacts_folder=Path(f'artifacts/on_images/deep_learning/v{version}')
log_file_path=artifacts_folder / 'experiments.parquet'

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = True  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

augment=True  # Augment data for training

BATCH_SIZE = 32

RANDOM_SEED = 42

# load_model=True
load_model=False

## Preprocessing

In [11]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [12]:
if rebalance_with_weights:
    print('using class weights')
else:
    print('not using class weights')

using class weights


In [13]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [14]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024700
min       0.008980
25%       0.018696
50%       0.031797
75%       0.054541
max       0.122185
Name: proportion, dtype: float64

In [15]:
y_train.value_counts().describe()

count     27.000000
mean     251.592593
std      167.786325
min       61.000000
25%      127.000000
50%      216.000000
75%      370.500000
max      830.000000
Name: count, dtype: float64

In [16]:
print(X_train.shape)

(6793, 31)


In [17]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [18]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, full_y_train=full_y_train, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights, augment=augment)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

24

In [19]:
new_preprocessors

{}

In [20]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [21]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False, augment=False)

## Model

In [22]:
from tensorflow import keras

### Load or create model

In [23]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    subversion = last_experiment.get('subversion', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    subversion = last_experiment.get('subversion', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model, base_model = define_model(pHash_vocab_size=len(preprocessors['hash'].categories_[0]), md5_vocab_size=len(preprocessors['hash'].categories_[1]), n_cols_tabular=n_cols_tabular, num_classes=27)


Création d'un nouveau modèle.


In [24]:
if not load_model:
    subversion = int(input(f"subversion (architecture)? (last: {subversion})"))

In [25]:
subversion

7

### Summary

In [26]:
# model.summary()

## Callbacks

### ModelCheckpoint

In [27]:
# # Pick an available filename to save a model.
# subversion=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{subversion}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     subversion+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{subversion}.h5')
# new_location_for_saving_model


In [28]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [29]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

In [30]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [31]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_loss', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='min',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [32]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [33]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [34]:
import datetime
tensor_board_folder = artifacts_folder / "tensorboard_logs"
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder / timestamp,
    histogram_freq=1 # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [35]:
import math
typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793

In [36]:
# max_epochs=13

# # Calculate expected duration
# available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
# available_minutes

In [37]:
# Pick max_epochs based on your available time
available_minutes=80

max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

38

### compilation and callbacks

In [38]:
learning_rate=0.001

In [39]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [40]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [41]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=80, max_epochs=38, champion_path=None ?

In [42]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=max(total_epochs_trained-1,0), callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 1/38


2025-10-04 15:16:57.276355: I external/local_xla/xla/service/service.cc:163] XLA service 0x747678024050 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-04 15:16:57.276373: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-04 15:16:57.968917: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-04 15:17:01.437856: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-04 15:17:09.583952: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:17:09.680113: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel 

424/425 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step - accuracy: 0.1628 - loss: 3.4850

2025-10-04 15:19:29.838554: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:19:29.933655: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:19:30.549507: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:19:30.649504: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:19:31.313634: E external/local_xla/xla/stream_

425/425 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step - accuracy: 0.1629 - loss: 3.4845

2025-10-04 15:21:14.992650: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:21:15.087285: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:21:15.672510: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:21:15.771945: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-04 15:21:16.393452: E external/local_xla/xla/stream_

425/425 ━━━━━━━━━━━━━━━━━━━━ 286s 510ms/step - accuracy: 0.2017 - loss: 3.2843 - val_accuracy: 0.3000 - val_loss: 2.8228 - learning_rate: 0.0010
Epoch 2/38
425/425 ━━━━━━━━━━━━━━━━━━━━ 162s 380ms/step - accuracy: 0.2971 - loss: 2.7933 - val_accuracy: 0.3948 - val_loss: 2.4404 - learning_rate: 0.0010
Epoch 3/38
425/425 ━━━━━━━━━━━━━━━━━━━━ 165s 389ms/step - accuracy: 0.3546 - loss: 2.5400 - val_accuracy: 0.4132 - val_loss: 2.2969 - learning_rate: 0.0010
Epoch 4/38
425/425 ━━━━━━━━━━━━━━━━━━━━ 163s 383ms/step - accuracy: 0.4028 - loss: 2.3729 - val_accuracy: 0.4429 - val_loss: 2.2424 - learning_rate: 0.0010
Epoch 5/38
425/425 ━━━━━━━━━━━━━━━━━━━━ 162s 380ms/step - accuracy: 0.4438 - loss: 2.2128 - val_accuracy: 0.4583 - val_loss: 2.1782 - learning_rate: 0.0010
Epoch 6/38
425/425 ━━━━━━━━━━━━━━━━━━━━ 162s 382ms/step - accuracy: 0.5244 - loss: 1.9787 - val_accuracy: 0.4740 - val_loss: 2.2098 - learning_rate: 0.0010
Epoch 7/38
425/425 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step - accuracy: 0.6041 - 

'total_minutes=29.11668768723806'

## Evaluation

In [43]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

source venv/bin/activate
tensorboard --logdir '/home/val/Documents/Dev/DataScientest/Rakuten/artifacts/on_images/deep_learning/v1/tensorboard_logs'


In [44]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['val_accuracy'] = max(model_history.history['val_accuracy'])
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.4739755094051361,
 'actual_epochs': 10,
 'minutes_per_epoch': 2.911668768723806}

In [45]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [46]:
#Takes 1m40
y_pred = model.predict(test_ds)
y_pred

1062/1062 ━━━━━━━━━━━━━━━━━━━━ 73s 64ms/step


array([[1.51783321e-03, 1.31342942e-02, 4.48737778e-02, ...,
        6.04126938e-02, 2.05303915e-03, 4.67846822e-03],
       [2.60861358e-03, 1.78131014e-02, 4.53158654e-02, ...,
        4.45025191e-02, 2.13759346e-03, 1.41645363e-02],
       [3.63303407e-04, 1.01875952e-02, 6.32312819e-02, ...,
        4.39730696e-02, 6.91277091e-04, 2.68491451e-03],
       ...,
       [3.37983698e-01, 1.39775872e-02, 2.09757200e-04, ...,
        7.47595986e-05, 5.83581507e-01, 1.35261880e-03],
       [5.54996636e-03, 9.49994102e-03, 1.14264973e-02, ...,
        2.25832723e-02, 2.56080576e-03, 1.61794433e-03],
       [2.02663261e-02, 2.28611361e-02, 2.79289223e-02, ...,
        4.85885181e-02, 1.14766555e-02, 4.41970816e-03]],
      shape=(16984, 27), dtype=float32)

In [47]:
from sklearn import metrics

In [48]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [49]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,60,1140,1160,1280,1281,1300,1320,1560,1920,2060,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,
10,323,22,0,0,16,4,0,0,0,4,0,1,150,59,1,7,0,6,0,30,0
40,43,188,0,3,56,18,0,38,1,8,0,6,68,28,4,9,7,13,0,9,3
50,8,11,0,9,7,35,0,111,4,6,1,14,4,7,9,14,8,85,0,1,2
60,4,9,9,0,4,4,0,98,0,1,0,10,2,3,2,6,0,11,0,0,3
1140,30,32,0,120,33,128,0,28,5,1,1,10,19,29,5,14,1,65,0,11,2
1160,4,24,1,5,693,6,0,2,0,0,0,0,34,13,4,2,0,3,0,0,0
1180,7,6,0,23,24,20,0,12,1,1,1,1,19,13,1,6,0,15,0,2,1
1280,10,21,0,60,26,330,0,176,16,12,12,107,13,20,4,17,15,116,0,10,9
1281,23,39,0,4,40,87,1,17,8,2,4,35,25,26,4,17,4,41,0,20,17


In [50]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.410350720845763, 0.0)

In [51]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [52]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/val/Documents/Dev/DataScientest/Rakuten/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

,precision,recall,f1-score,support
10,0.376019,0.518459,0.435897,623.000000
40,0.411379,0.374502,0.392075,502.000000
50,0.000000,0.000000,0.000000,336.000000
60,0.642857,0.054217,0.100000,166.000000
1140,0.428571,0.224719,0.294840,534.000000
1160,0.651929,0.876106,0.747573,791.000000
1180,0.000000,0.000000,0.000000,153.000000
1280,0.232231,0.338809,0.275574,974.000000
1281,0.500000,0.002415,0.004808,414.000000
1300,0.395921,0.654113,0.493274,1009.000000


In [53]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.371173,0.336614,0.311434,629.037037
std,0.234475,0.309475,0.261518,420.759339
min,0.000000,0.000000,0.000000,153.000000
25%,0.214292,0.002210,0.004400,310.000000
50%,0.411498,0.270270,0.294840,534.000000
75%,0.508867,0.578626,0.508974,953.500000
max,0.739247,0.876106,0.747573,2042.000000


In [54]:
# negative correlation between support and another measure would suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.641462,0.724162,0.065197
recall,0.641462,1.000000,0.975884,0.104272
f1-score,0.724162,0.975884,1.000000,0.097333
support,0.065197,0.104272,0.097333,1.000000


In [55]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.410350720845763

In [56]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.4739755094051361,
 'actual_epochs': 10,
 'minutes_per_epoch': 2.911668768723806,
 'weighted_avg_f1_score': 0.410350720845763,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.2615179277578862)}

## Update tracker

In [57]:
tracker['comment']="Unfreezed base model. Batch size from 32 to 16 because memory error."
tracker['comment']

'Unfreezed base model. Batch size from 32 to 16 because memory error.'

In [ ]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Nouveau champion ! Sauvegarde du modèle.")
    keep_candidate=True

    best_epoch_in_session_idx = np.argmin(model_history.history['val_loss'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_loss = model_history.history['val_loss'][best_epoch_in_session_idx]


    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_sv-{subversion}_epoch_index-{best_epoch_global:02d}_val_loss-{best_val_loss:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_filename)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print("Effacement de l'ancien modèle de la même subversion {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le champion.")
    keep_candidate=False


Nouveau champion ! Sauvegarde du modèle.
best_model_sv-7_epoch_index-04_val_loss-2.1782_f1-0.4104.keras


In [59]:
tracker['epoch_index'] = best_epoch_global
tracker['total_epochs'] = total_epochs_trained + best_epoch_global + 1

In [60]:
to_track=['subversion','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate','timestamp']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model, base_model)

In [61]:
tracker

{'X_train.shape[0]': 6793,
 'val_accuracy': 0.4739755094051361,
 'actual_epochs': 10,
 'minutes_per_epoch': 2.911668768723806,
 'weighted_avg_f1_score': 0.410350720845763,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.2615179277578862),
 'comment': 'Unfreezed base model. Batch size from 32 to 16 because memory error.',
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-7_epoch_index-04_val_loss-2.1782_f1-0.4104.keras',
 'epoch_index': np.int64(4),
 'total_epochs': np.int64(5),
 'subversion': 7,
 'max_epochs': 38,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 16,
 'learning_rate': 0.001,
 'timestamp': '20251004-151611',
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16}}

In [62]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [63]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [64]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [65]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, loaded_model=load_model, log_file_path=log_file_path)

Log pour l'expérience subversion 7 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet .


## Show tracking logs

In [66]:
pd.set_option('max_colwidth', None)

In [67]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,timestamp,comment,best_model_path
0,2,False,6793,32,1.998601,10,0.001,0.569713,0.528593,0.0,0.247884,256_128_64_32,16_16,None,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.,artifacts/on_images/deep_learning/v1/best_model_sv-2_epochs-10_f1-0.5286.keras
1,2,False,20380,32,3.867331,8,0.001,0.584609,0.563841,0.0,0.239801,256_128_64_32,16_16,None,Increased frac from 0.1 to 0.3.,artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras
2,3,False,6793,32,2.024143,10,0.001,0.566710,0.527659,0.0,0.243439,128_64_32_16,8_8,None,"Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings. Training curve is less steep and validation curve goes higher, which suggests overfitting has been reduced.",artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-04_val_loss-1.8077_f1-0.5277.keras
3,4,True,6793,32,1.954997,13,0.001,0.541922,0.490818,0.0,0.270646,128_64_32_16,8_8,None,"Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. Less overfitting: performance is better on validation than on train until epoch 7, and train performance curve is less steep.",artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-07_val_loss-1.8789_f1-0.4908.keras
4,4,True,20380,32,3.115025,4,0.001,0.562529,0.499731,0.0,0.276383,128_64_32_16,8_8,None,Increased frac from .1 to .3.,artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-03_val_loss-1.7963_f1-0.4997.keras
5,5,True,20380,32,3.080263,3,0.001,0.566945,0.506920,0.0,0.255305,128_64_32_16,8_8,None,Removed dropout layer before softmax. Performance better but more overfitting.,artifacts/on_images/deep_learning/v1/best_model_sv-5_epoch_index-02_val_loss-1.7414_f1-0.5069.keras
6,6,True,6793,32,1.781595,6,0.001,0.558585,0.537182,0.0,0.223538,256_128_64_32,16_16,None,Frac back to .1. Reverted layers/embeddings to higher sizes and uncommented Dropout before softmax.,artifacts/on_images/deep_learning/v1/best_model_sv-6_epoch_index-05_val_loss-1.9153_f1-0.5372.keras
7,7,True,6793,16,2.911669,5,0.001,0.473976,0.410351,0.0,0.261518,256_128_64_32,16_16,20251004-151611,Unfreezed base model. Batch size from 32 to 16 because memory error.,artifacts/on_images/deep_learning/v1/best_model_sv-7_epoch_index-04_val_loss-2.1782_f1-0.4104.keras


In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
